# Module 06: Supervised Learning (Algorithms)


## Module outline

* **Step 1: Parametric & Instance-Based Models**
  * Simple & Multiple Linear Regression: Ordinary Least Squares (OLS) assumptions (linearity, homoscedasticity, independence, normality of errors).
  * Polynomial Regression: Modeling non-linear systems.
  * Logistic Regression: Sigmoid transformation, log-odds, maximum likelihood estimation, binary vs. multinomial (one-vs-rest / softmax) classification.
  * K-Nearest Neighbors (KNN): Distance metrics (Euclidean, Manhattan, Minkowski, Cosine), choosing $k$, and the Curse of Dimensionality.
  * Naive Bayes: Class-conditional independence assumption, Bayes' Theorem, Gaussian Naive Bayes, Multinomial Naive Bayes, and Laplace Smoothing.
* **Step 2: Support Vector Machines & Decision Trees**
  * Support Vector Machines (SVM): Maximize margin width, Support Vectors, Hard vs. Soft Margin (penalty parameter $C$ and slack variables), Kernel Trick (Linear, Polynomial, RBF, Sigmoid), and Mercer's Theorem.
  * Decision Trees: Classification and Regression Trees (CART), split criteria metrics (Gini Impurity, Information Gain/Shannon Entropy, Variance Reduction), stopping criteria (max_depth, min_samples_split, min_samples_leaf), and Minimal Cost-Complexity Pruning.
* **Step 3: Bagging & Ensemble Learning**
  * Ensemble Philosophy: Combining weak learners to reduce variance (Bagging) or bias (Boosting).
  * Bagging (Bootstrap Aggregating): Out-of-Bag (OOB) error estimation.
  * Random Forest: Feature subspace sampling, voting/averaging, and structural hyperparameter control (n_estimators, max_features).
  * Extra Trees (Extremely Randomized Trees): Random split-point selection to reduce variance.
* **Step 4: Advanced Boosting Ecosystem**
  * Adaptive Boosting (AdaBoost): Exponential loss optimization, sequential sample weight updating, and estimator weighting.
  * Gradient Boosting Machines (GBM): Loss function optimization via gradient descent, pseudo-residuals, learning rate (shrinkage), and base-learner scaling.
  * **Modern Gradient Boosting Architectures:**
    * **XGBoost (Extreme Gradient Boosting):** Regularized objective function (including L1/L2 on leaf weights), second-order Taylor expansion approximation of loss, sparsity-aware split finding (native missing value handling), and block structure parallelization.
    * **LightGBM (Light Gradient Boosting Machine):** Gradient-based One-Side Sampling (GOSS), Exclusive Feature Bundling (EFB), leaf-wise (best-first) tree growth, and histogram-based split optimization.
    * **CatBoost (Categorical Boosting):** Ordered Boosting to prevent target leakage, symmetric trees (oblivious trees) for fast inference, and native categorical feature handling with on-the-fly target statistics.
  * Stacking & Blending: Multi-layer model architectures using out-of-fold predictions to train a meta-learner.


## Example Dataset

We will use the same employee dataset, but with two clear tasks:

- **Regression task:** predict `Monthly Salary`.
- **Classification task:** predict `Attrition (Left?)`, which means whether the employee left the company.

For teaching algorithms, each algorithm gets its own explanation, code, and result reading.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

# Same 10-employee teaching dataset used across the modules.
data = {
    'Employee ID': ['Emp 1', 'Emp 2', 'Emp 3', 'Emp 4', 'Emp 5', 'Emp 6', 'Emp 7', 'Emp 8', 'Emp 9', 'Emp 10'],
    'Department': ['Sales', 'R&D', 'HR', 'R&D', 'Sales', 'R&D', 'HR', 'Sales', 'R&D', 'Sales'],
    'Performance Rating': ['Low', 'Medium', 'Medium', 'High', 'High', 'Low', 'High', 'Medium', 'High', 'Medium'],
    'Years at Company': [1, 3, 2, 5, 4, 1, 8, 3, 10, 3],
    'Monthly Salary': [3000, 4500, 4000, 6000, 5500, 3200, 7000, 4800, 12000, 5000],
    'Attrition (Left?)': ['Yes (1)', 'No (0)', 'No (0)', 'No (0)', 'No (0)', 'Yes (1)', 'No (0)', 'No (0)', 'No (0)', 'Yes (1)']
}

df = pd.DataFrame(data)
df['Left_Binary'] = df['Attrition (Left?)'].apply(lambda x: 1 if 'Yes' in x else 0)
df['Performance_Ordinal'] = df['Performance Rating'].map({'Low': 0, 'Medium': 1, 'High': 2})

display(df)


,Employee ID,Department,Performance Rating,Years at Company,Monthly Salary,Attrition (Left?),Left_Binary,Performance_Ordinal
0,Emp 1,Sales,Low,1,3000,Yes (1),1,0
1,Emp 2,R&D,Medium,3,4500,No (0),0,1
2,Emp 3,HR,Medium,2,4000,No (0),0,1
3,Emp 4,R&D,High,5,6000,No (0),0,2
4,Emp 5,Sales,High,4,5500,No (0),0,2
5,Emp 6,R&D,Low,1,3200,Yes (1),1,0
6,Emp 7,HR,High,8,7000,No (0),0,2
7,Emp 8,Sales,Medium,3,4800,No (0),0,1
8,Emp 9,R&D,High,10,12000,No (0),0,2
9,Emp 10,Sales,Medium,3,5000,Yes (1),1,1


## Module 6: Supervised Learning Algorithms

Supervised learning means the model learns from examples where the answer is already known.

```text
Input features X -> Model -> Prediction y
```

Examples:

```text
Years at Company -> predict Monthly Salary          regression
Years + Salary -> predict Left or Stayed            classification
```

In this module, do not learn algorithm names as a list. Learn each algorithm like this:

```text
What it does -> where/why we use it -> code -> how to read the output
```

### Algorithm 1 - Simple Linear Regression

- **Definition:** Linear regression predicts a number using a straight-line relationship.
- **Where we use it:** Salary, price, demand, marks, revenue, or any numerical target.
- **Why we use it:** It is simple and easy to explain.

Here we predict `Monthly Salary` using only `Years at Company`.

Simple equation:

```text
Predicted Salary = intercept + weight * Years at Company
```

Meaning:

- **Intercept:** predicted salary when years is 0.
- **Weight:** how much salary changes when years increases by 1.

This is a regression algorithm because the output is a number.

#### Visual intuition

**Linear regression best-fit line**

This shows the main idea of linear regression: a straight line is chosen to stay close to the data points.

![Linear regression best-fit line](images/module06/linear_regression.png)

*Image source: [File:Linear-regression.svg](https://commons.wikimedia.org/wiki/File:Linear-regression.svg) by Qef; license: Public domain.*


In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

X = df[['Years at Company']]
y = df['Monthly Salary']

linear_model = LinearRegression()
linear_model.fit(X, y)
predicted_salary = linear_model.predict(X)

linear_results = pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Years at Company': df['Years at Company'],
    'Actual Salary': y,
    'Predicted Salary': predicted_salary.round(0),
    'Error': (predicted_salary - y).round(0)
})

print('Linear Regression equation:')
print(f'Predicted Salary = {linear_model.intercept_:.2f} + {linear_model.coef_[0]:.2f} * Years at Company')
print(f'MAE: {mean_absolute_error(y, predicted_salary):.2f}')
print(f'R-squared: {r2_score(y, predicted_salary):.3f}')

display(linear_results)


Linear Regression equation:
Predicted Salary = 2161.54 + 834.62 * Years at Company
MAE: 467.69
R-squared: 0.901


,Employee ID,Years at Company,Actual Salary,Predicted Salary,Error
0,Emp 1,1,3000,2996.0,-4.0
1,Emp 2,3,4500,4665.0,165.0
2,Emp 3,2,4000,3831.0,-169.0
3,Emp 4,5,6000,6335.0,335.0
4,Emp 5,4,5500,5500.0,0.0
5,Emp 6,1,3200,2996.0,-204.0
6,Emp 7,8,7000,8838.0,1838.0
7,Emp 8,3,4800,4665.0,-135.0
8,Emp 9,10,12000,10508.0,-1492.0
9,Emp 10,3,5000,4665.0,-335.0


**How to read the result:**

The coefficient tells how salary changes with one more year at the company, according to this simple line. The error column shows where the line is wrong for each employee.

This model is easy to explain, but it may underfit if salary does not follow a straight-line pattern. For example, one senior employee with salary `12000` may not be captured well by one straight line.

### Algorithm 2 - Multiple Linear Regression

- **Definition:** Multiple linear regression predicts a number using more than one input feature.
- **Where we use it:** When salary may depend on tenure, performance, department, experience, and other factors.
- **Why we use it:** One feature is often not enough.

Here we predict `Monthly Salary` using:

```text
Years at Company + Performance Rating
```

Performance is converted to ordered numbers:

```text
Low = 0, Medium = 1, High = 2
```

In [3]:
X = df[['Years at Company', 'Performance_Ordinal']]
y = df['Monthly Salary']

multi_model = LinearRegression()
multi_model.fit(X, y)
multi_pred = multi_model.predict(X)

print('Multiple Linear Regression equation:')
print(f'Predicted Salary = {multi_model.intercept_:.2f} + {multi_model.coef_[0]:.2f} * Years + {multi_model.coef_[1]:.2f} * Performance')
print(f'MAE: {mean_absolute_error(y, multi_pred):.2f}')
print(f'R-squared: {r2_score(y, multi_pred):.3f}')

display(pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Years at Company': df['Years at Company'],
    'Performance Rating': df['Performance Rating'],
    'Actual Salary': y,
    'Predicted Salary': multi_pred.round(0),
    'Error': (multi_pred - y).round(0)
}))


Multiple Linear Regression equation:
Predicted Salary = 2315.43 + 936.81 * Years + -468.88 * Performance
MAE: 479.05
R-squared: 0.908


,Employee ID,Years at Company,Performance Rating,Actual Salary,Predicted Salary,Error
0,Emp 1,1,Low,3000,3252.0,252.0
1,Emp 2,3,Medium,4500,4657.0,157.0
2,Emp 3,2,Medium,4000,3720.0,-280.0
3,Emp 4,5,High,6000,6062.0,62.0
4,Emp 5,4,High,5500,5125.0,-375.0
5,Emp 6,1,Low,3200,3252.0,52.0
6,Emp 7,8,High,7000,8872.0,1872.0
7,Emp 8,3,Medium,4800,4657.0,-143.0
8,Emp 9,10,High,12000,10746.0,-1254.0
9,Emp 10,3,Medium,5000,4657.0,-343.0


**How to read the result:**

The model now has one weight for tenure and one weight for performance. This lets it use more information than simple linear regression.

Be careful: converting `Low`, `Medium`, `High` to `0`, `1`, `2` assumes the order is meaningful. That is acceptable here because performance rating has a natural order.

### Algorithm 3 - Polynomial Regression

- **Definition:** Polynomial regression adds curved terms such as `Years^2`.
- **Where we use it:** When the relationship is not a straight line.
- **Why we use it:** It lets a linear model fit curves.

Example:

```text
Predicted Salary = intercept + w1 * Years + w2 * Years^2
```

This can fit curves better than a straight line, but high-degree polynomial models can overfit.

#### Visual intuition

**Flexible curve and overfitting idea**

This shows why polynomial models must be controlled: a very flexible curve can start following noise instead of the real pattern.

![Flexible curve and overfitting idea](images/module06/polynomial_overfitting.png)

*Image source: [File:Overfitting.svg](https://commons.wikimedia.org/wiki/File:Overfitting.svg) by Chabacano; license: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0).*


In [4]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

X = df[['Years at Company']]
y = df['Monthly Salary']

poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])
poly_model.fit(X, y)
poly_pred = poly_model.predict(X)

print('Polynomial Regression, degree = 2')
print(f'MAE: {mean_absolute_error(y, poly_pred):.2f}')
print(f'R-squared: {r2_score(y, poly_pred):.3f}')

display(pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Years at Company': df['Years at Company'],
    'Actual Salary': y,
    'Predicted Salary': poly_pred.round(0),
    'Error': (poly_pred - y).round(0)
}))


Polynomial Regression, degree = 2
MAE: 490.30
R-squared: 0.924


,Employee ID,Years at Company,Actual Salary,Predicted Salary,Error
0,Emp 1,1,3000,3480.0,480.0
1,Emp 2,3,4500,4460.0,-40.0
2,Emp 3,2,4000,3919.0,-81.0
3,Emp 4,5,6000,5848.0,-152.0
4,Emp 5,4,5500,5103.0,-397.0
5,Emp 6,1,3200,3480.0,280.0
6,Emp 7,8,7000,8692.0,1692.0
7,Emp 8,3,4800,4460.0,-340.0
8,Emp 9,10,12000,11097.0,-903.0
9,Emp 10,3,5000,4460.0,-540.0


**How to read the result:**

Compare this with simple linear regression. If the polynomial model reduces error, it may be capturing a curve. But if the degree becomes too high, it may memorize the training data and fail on new employees.

### Algorithm 4 - Logistic Regression

- **Definition:** Logistic regression predicts the probability of a class.
- **Where we use it:** Yes/no tasks like `Left` vs `Stayed`.
- **Why we use it:** It is a strong, interpretable classification baseline.

Here we predict whether an employee left the company.

```text
Output = probability of leaving
```

If probability is greater than a threshold, commonly `0.5`, we predict class `1`.

Important terms:

- **Sigmoid:** converts any score into a probability between 0 and 1.
- **Log-odds:** the scale logistic regression learns internally.
- **Binary classification:** two classes.
- **Multinomial classification:** more than two classes, often using softmax.

#### Visual intuition

**Logistic sigmoid curve**

This shows the sigmoid curve used by logistic regression to convert a score into a probability between 0 and 1.

![Logistic sigmoid curve](images/module06/logistic_sigmoid.png)

*Image source: [File:Logistic-curve.svg](https://commons.wikimedia.org/wiki/File:Logistic-curve.svg) by Qef (talk); license: Public domain.*


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']

logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])
logistic_model.fit(X, y)
left_probability = logistic_model.predict_proba(X)[:, 1]
left_prediction = logistic_model.predict(X)

print('Logistic Regression classification metrics on teaching data:')
print(f'Accuracy: {accuracy_score(y, left_prediction):.3f}')
print(f'Precision: {precision_score(y, left_prediction, zero_division=0):.3f}')
print(f'Recall: {recall_score(y, left_prediction, zero_division=0):.3f}')
print(f'F1: {f1_score(y, left_prediction, zero_division=0):.3f}')

display(pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Years at Company': df['Years at Company'],
    'Monthly Salary': df['Monthly Salary'],
    'Actual Left?': y,
    'Probability of Leaving': left_probability.round(3),
    'Predicted Left?': left_prediction
}))


Logistic Regression classification metrics on teaching data:
Accuracy: 0.900
Precision: 1.000
Recall: 0.667
F1: 0.800


,Employee ID,Years at Company,Monthly Salary,Actual Left?,Probability of Leaving,Predicted Left?
0,Emp 1,1,3000,1,0.568,1
1,Emp 2,3,4500,0,0.338,0
2,Emp 3,2,4000,0,0.436,0
3,Emp 4,5,6000,0,0.166,0
4,Emp 5,4,5500,0,0.231,0
5,Emp 6,1,3200,1,0.556,1
6,Emp 7,8,7000,0,0.061,0
7,Emp 8,3,4800,0,0.322,0
8,Emp 9,10,12000,0,0.011,0
9,Emp 10,3,5000,1,0.312,0


**How to read the result:**

The probability column is the most important output. A value like `0.70` means the model estimates a 70% chance of leaving for that row.

This tiny dataset is only for learning mechanics. In real use, we would evaluate on unseen data before trusting the probabilities.

### Algorithm 5 - K-Nearest Neighbors (KNN)

- **Definition:** KNN predicts using the closest training rows.
- **Where we use it:** When similar rows should have similar answers.
- **Why we use it:** The idea is simple and visual.

For attrition prediction:

```text
Find employees most similar in Years and Salary.
Use their classes to vote.
```

Important settings:

- `k`: number of neighbors.
- distance metric: how closeness is measured.

Distance examples:

- **Euclidean:** straight-line distance.
- **Manhattan:** city-block distance.
- **Cosine:** angle/direction similarity.

KNN usually needs scaling because salary numbers are much larger than years.

#### Visual intuition

**K-nearest neighbors classification**

This shows the KNN idea: the unknown point is classified by looking at nearby points.

![K-nearest neighbors classification](images/module06/knn.png)

*Image source: [File:KnnClassification.svg](https://commons.wikimedia.org/wiki/File:KnnClassification.svg) by Natana Jadri AnAj; license: [CC BY-SA 3.0](http://creativecommons.org/licenses/by-sa/3.0/).*


In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']

knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=3, metric='euclidean'))
])
knn_model.fit(X, y)
knn_pred = knn_model.predict(X)
knn_prob = knn_model.predict_proba(X)[:, 1]

print('KNN with k = 3')
print(f'Accuracy on teaching data: {accuracy_score(y, knn_pred):.3f}')

display(pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Actual Left?': y,
    'KNN Probability of Leaving': knn_prob.round(3),
    'KNN Predicted Left?': knn_pred
}))


KNN with k = 3
Accuracy on teaching data: 0.900


,Employee ID,Actual Left?,KNN Probability of Leaving,KNN Predicted Left?
0,Emp 1,1,0.667,1
1,Emp 2,0,0.333,0
2,Emp 3,0,0.000,0
3,Emp 4,0,0.333,0
4,Emp 5,0,0.333,0
5,Emp 6,1,0.667,1
6,Emp 7,0,0.000,0
7,Emp 8,0,0.333,0
8,Emp 9,0,0.000,0
9,Emp 10,1,0.333,0


**How to read the result:**

KNN does not learn a formula like linear regression. It stores the training examples and checks nearby rows when predicting.

Small `k` can overfit. Large `k` can smooth too much and underfit. In high-dimensional data, distances become less reliable; this is called the curse of dimensionality.

### Algorithm 6 - Naive Bayes

- **Definition:** Naive Bayes is a probabilistic classifier based on Bayes' theorem.
- **Where we use it:** Text classification, spam detection, and fast baseline models.
- **Why we use it:** It is very fast and often works well even with simple assumptions.

Bayes idea:

```text
Use the evidence from features to update the probability of each class.
```

The word **naive** means it assumes features are conditionally independent inside each class.

Types:

- **Gaussian Naive Bayes:** numerical features.
- **Multinomial Naive Bayes:** count features, common in text.
- **Laplace smoothing:** prevents zero probability when a word/category was unseen in training.

#### Visual intuition

**Bayes theorem probability tree**

This tree diagram shows the Bayes idea more simply: probabilities move through branches, and evidence changes which branch is likely.

![Bayes theorem probability tree](images/module06/naive_bayes.png)

*Image source: [File:Bayes theorem example tree.svg](https://commons.wikimedia.org/wiki/File:Bayes_theorem_example_tree.svg) by Gnathan87; license: [CC0](http://creativecommons.org/publicdomain/zero/1.0/deed.en).*


In [7]:
from sklearn.naive_bayes import GaussianNB

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']

nb_model = GaussianNB()
nb_model.fit(X, y)
nb_pred = nb_model.predict(X)
nb_prob = nb_model.predict_proba(X)[:, 1]

print('Gaussian Naive Bayes')
print(f'Accuracy on teaching data: {accuracy_score(y, nb_pred):.3f}')

display(pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Actual Left?': y,
    'NB Probability of Leaving': nb_prob.round(3),
    'NB Predicted Left?': nb_pred
}))


Gaussian Naive Bayes
Accuracy on teaching data: 0.700


,Employee ID,Actual Left?,NB Probability of Leaving,NB Predicted Left?
0,Emp 1,1,0.929,1
1,Emp 2,0,0.598,1
2,Emp 3,0,0.895,1
3,Emp 4,0,0.000,0
4,Emp 5,0,0.026,0
5,Emp 6,1,0.933,1
6,Emp 7,0,0.000,0
7,Emp 8,0,0.495,0
8,Emp 9,0,0.000,0
9,Emp 10,1,0.413,0


**How to read the result:**

Naive Bayes gives class probabilities. With Gaussian Naive Bayes, the model assumes each numerical feature follows a bell-shaped distribution inside each class.

This is a strong baseline idea, but the independence assumption may be unrealistic for salary and tenure because those columns can be related.

### Algorithm 7 - Support Vector Machine (SVM)

- **Definition:** SVM finds a boundary that separates classes with the widest margin.
- **Where we use it:** Classification problems where a clear separating boundary may exist.
- **Why we use it:** A wider margin can improve generalization.

Important terms:

- **Margin:** distance between boundary and nearest points.
- **Support vectors:** points closest to the boundary; they define the boundary.
- **Hard margin:** no mistakes allowed.
- **Soft margin:** allows some mistakes.
- **C:** controls how strongly mistakes are punished.

Kernel trick:

- **Linear kernel:** straight boundary.
- **Polynomial kernel:** curved polynomial boundary.
- **RBF kernel:** flexible curved boundary.
- **Sigmoid kernel:** less common, neural-network-like shape.

Mercer's theorem is the mathematical rule for valid kernels. At this stage, use standard library kernels.

#### Visual intuition

**SVM margin and support vectors**

This shows the SVM margin: the boundary is chosen using the closest important points.

![SVM margin and support vectors](images/module06/svm.png)

*Image source: [File:SVM margins.svg](https://commons.wikimedia.org/wiki/File:SVM_margins.svg) by DJGulp3; license: Public domain.*


In [8]:
from sklearn.svm import SVC

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']

svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', C=1.0, probability=True, random_state=42))
])
svm_model.fit(X, y)
svm_pred = svm_model.predict(X)
svm_prob = svm_model.predict_proba(X)[:, 1]

print('SVM with RBF kernel')
print(f'Accuracy on teaching data: {accuracy_score(y, svm_pred):.3f}')

display(pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Actual Left?': y,
    'SVM Probability of Leaving': svm_prob.round(3),
    'SVM Predicted Left?': svm_pred
}))


SVM with RBF kernel
Accuracy on teaching data: 0.700


,Employee ID,Actual Left?,SVM Probability of Leaving,SVM Predicted Left?
0,Emp 1,1,0.083,0
1,Emp 2,0,0.390,0
2,Emp 3,0,0.217,0
3,Emp 4,0,0.517,0
4,Emp 5,0,0.517,0
5,Emp 6,1,0.090,0
6,Emp 7,0,0.416,0
7,Emp 8,0,0.416,0
8,Emp 9,0,0.416,0
9,Emp 10,1,0.430,0


**How to read the result:**

SVM focuses heavily on boundary-defining points. `C` controls the tradeoff between a clean boundary and allowing mistakes.

The RBF kernel can create curved boundaries. It is flexible, so validation is important to avoid overfitting.

### Algorithm 8 - Decision Tree

- **Definition:** A decision tree predicts by asking a sequence of yes/no questions.
- **Where we use it:** Tabular regression and classification.
- **Why we use it:** Trees are easy to explain.

Example:

```text
if Years at Company <= 2:
    predict Left
else:
    ask another question
```

Split criteria:

- **Gini impurity:** measures class mixing.
- **Entropy / information gain:** measures uncertainty reduction.
- **Variance reduction:** used for regression trees.

Stopping controls:

- `max_depth`
- `min_samples_split`
- `min_samples_leaf`

Pruning removes branches that add complexity without enough value.

#### Visual intuition

**Recursive splitting in a decision tree**

This shows the core decision-tree idea: repeated threshold splits divide the feature space, and those splits become tree branches.

![Recursive splitting in a decision tree](images/module06/decision_tree.png)

*Image source: [File:Recursive Splitting.png](https://commons.wikimedia.org/wiki/File:Recursive_Splitting.png) by Rossc0827; license: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0).*


In [9]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X, y)
tree_pred = tree_model.predict(X)

print('Decision Tree Classifier')
print(f'Accuracy on teaching data: {accuracy_score(y, tree_pred):.3f}')
print('\nTree rules:')
print(export_text(tree_model, feature_names=list(X.columns)))

display(pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Actual Left?': y,
    'Tree Predicted Left?': tree_pred
}))


Decision Tree Classifier
Accuracy on teaching data: 1.000

Tree rules:
|--- Years at Company <= 1.50
|   |--- class: 1
|--- Years at Company >  1.50
|   |--- Monthly Salary <= 5250.00
|   |   |--- Monthly Salary <= 4900.00
|   |   |   |--- class: 0
|   |   |--- Monthly Salary >  4900.00
|   |   |   |--- class: 1
|   |--- Monthly Salary >  5250.00
|   |   |--- class: 0



,Employee ID,Actual Left?,Tree Predicted Left?
0,Emp 1,1,1
1,Emp 2,0,0
2,Emp 3,0,0
3,Emp 4,0,0
4,Emp 5,0,0
5,Emp 6,1,1
6,Emp 7,0,0
7,Emp 8,0,0
8,Emp 9,0,0
9,Emp 10,1,1


**How to read the result:**

The printed rules show exactly how the tree makes decisions. This is why trees are easy to teach.

But a deep tree can memorize training rows. That is why `max_depth`, `min_samples_leaf`, and pruning matter.

### Algorithm 9 - Bagging, Random Forest, and Extra Trees

#### Bagging

Bagging means training many models on bootstrap samples and combining their predictions.

```text
many trees -> vote/average -> final prediction
```

It mainly reduces variance.

#### Random Forest

Random Forest is bagging with decision trees plus random feature selection.

Important settings:

- `n_estimators`: number of trees.
- `max_features`: number of features considered at a split.
- `max_depth`: tree depth.

#### Out-of-Bag Error

Because each tree uses a bootstrap sample, some rows are left out for that tree. These are out-of-bag rows and can estimate performance.

#### Extra Trees

Extra Trees adds even more randomness by choosing split points more randomly. This can reduce variance and make training faster.

#### Visual intuition

**Bagging ensemble flow**

This shows bagging: many models are trained on bootstrapped data and then aggregated.

![Bagging ensemble flow](images/module06/bagging.png)

*Image source: [File:Ensemble Bagging.svg](https://commons.wikimedia.org/wiki/File:Ensemble_Bagging.svg) by Sirakorn; license: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0).*

**Decision tree vs random forest**

This shows the random forest idea clearly: one decision tree is compared with a forest made from many trees.

![Decision tree vs random forest](images/module06/random_forest.png)

*Image source: [File:Decision Tree vs. Random Forest.png](https://commons.wikimedia.org/wiki/File:Decision_Tree_vs._Random_Forest.png) by Jeremybeauchamp; license: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0).*


In [10]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
    n_estimators=30,
    random_state=42
)
forest = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42, oob_score=True, bootstrap=True)
extra = ExtraTreesClassifier(n_estimators=50, max_depth=3, random_state=42)

models = {'Bagging': bagging, 'Random Forest': forest, 'Extra Trees': extra}
rows = []
for name, model in models.items():
    model.fit(X, y)
    pred = model.predict(X)
    row = {'Model': name, 'Accuracy on teaching data': accuracy_score(y, pred)}
    if hasattr(model, 'oob_score_'):
        row['OOB Score'] = model.oob_score_
    rows.append(row)

display(pd.DataFrame(rows))


,Model,Accuracy on teaching data,OOB Score
0,Bagging,0.9,NaN
1,Random Forest,1.0,0.6
2,Extra Trees,0.9,NaN


**How to read the result:**

All three methods combine many trees. Random Forest and Extra Trees usually generalize better than one deep tree because they average many different trees.

The out-of-bag score is an internal estimate from rows not used by each tree's bootstrap sample.

### Algorithm 10 - Boosting: AdaBoost and Gradient Boosting

Boosting trains models sequentially.

```text
Model 1 makes mistakes
Model 2 focuses on those mistakes
Model 3 continues correcting
```

#### AdaBoost

- Increases weight on wrongly predicted rows.
- Combines weak learners into a stronger model.

#### Gradient Boosting

- Each new model learns the remaining error, called pseudo-residuals.
- Uses a learning rate to control how strongly each tree contributes.

Important settings:

- `n_estimators`: number of boosting rounds.
- `learning_rate`: contribution of each new learner.
- tree depth: complexity of each weak learner.

#### Visual intuition

**Boosting ensemble flow**

This shows boosting: models are trained sequentially, with later models focusing more on earlier mistakes.

![Boosting ensemble flow](images/module06/boosting.png)

*Image source: [File:Ensemble Boosting.svg](https://commons.wikimedia.org/wiki/File:Ensemble_Boosting.svg) by Sirakorn; license: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0).*


In [11]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=20,
    learning_rate=0.5,
    random_state=42
)
gbm = GradientBoostingClassifier(n_estimators=30, learning_rate=0.1, max_depth=2, random_state=42)

rows = []
for name, model in {'AdaBoost': ada, 'Gradient Boosting': gbm}.items():
    model.fit(X, y)
    pred = model.predict(X)
    rows.append({'Model': name, 'Accuracy on teaching data': accuracy_score(y, pred)})

display(pd.DataFrame(rows))


,Model,Accuracy on teaching data
0,AdaBoost,1.0
1,Gradient Boosting,1.0


**How to read the result:**

Boosting can fit complex patterns, but it can overfit if there are too many boosting rounds or if each tree is too complex.

`learning_rate` and `n_estimators` work together. A smaller learning rate often needs more estimators.

### Algorithm 11 - Modern Boosting: XGBoost, LightGBM, and CatBoost

These are advanced gradient boosting libraries. They are commonly used for tabular machine learning.

#### XGBoost

- Adds strong regularization.
- Uses second-order loss information.
- Handles sparse data efficiently.
- Can learn default directions for missing values.

#### LightGBM

- Uses histogram-based split finding.
- Grows trees leaf-wise.
- Uses GOSS to keep important high-gradient rows.
- Uses EFB to bundle sparse exclusive features.

#### CatBoost

- Handles categorical features natively.
- Uses ordered boosting to reduce target leakage.
- Uses symmetric trees for faster prediction.

These libraries are powerful, but the basic idea is still boosting:

```text
build many learners sequentially, each one improving the previous model
```

### Algorithm 12 - Stacking and Blending

Stacking combines different model predictions using another model.

```text
Logistic Regression prediction
KNN prediction
Random Forest prediction
        -> meta-model -> final prediction
```

- **Base models:** first-level models.
- **Meta-model:** model trained on base-model predictions.
- **Out-of-fold predictions:** used to avoid leakage when training the meta-model.

Blending is similar, but usually uses a holdout validation set instead of full out-of-fold predictions.

In [12]:
from sklearn.ensemble import StackingClassifier

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']

stacking = StackingClassifier(
    estimators=[
        ('logistic', Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=42))])),
        ('knn', Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=3))])),
        ('forest', RandomForestClassifier(n_estimators=20, random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=2
)
stacking.fit(X, y)
stack_pred = stacking.predict(X)

print('Stacking Classifier')
print(f'Accuracy on teaching data: {accuracy_score(y, stack_pred):.3f}')

display(pd.DataFrame({
    'Employee ID': df['Employee ID'],
    'Actual Left?': y,
    'Stacking Predicted Left?': stack_pred
}))


Stacking Classifier
Accuracy on teaching data: 0.700


,Employee ID,Actual Left?,Stacking Predicted Left?
0,Emp 1,1,0
1,Emp 2,0,0
2,Emp 3,0,0
3,Emp 4,0,0
4,Emp 5,0,0
5,Emp 6,1,0
6,Emp 7,0,0
7,Emp 8,0,0
8,Emp 9,0,0
9,Emp 10,1,0


**How to read the result:**

Stacking is not a beginner baseline. Use it after understanding individual models.

The key risk is leakage: the meta-model should learn from out-of-fold predictions, not predictions made on the same rows used to train each base model.

### Module 6: Outline Concepts - Deep Explanation Checklist

This section explains the important outline terms that should not be skipped. The algorithm sections above show implementation; this section makes the internal concepts clear.

#### Linear Regression and OLS Assumptions

**OLS means Ordinary Least Squares.** It chooses coefficients that make the squared prediction errors as small as possible.

```text
error = actual value - predicted value
OLS minimizes sum(error^2)
```

OLS assumptions are not just theory words:

| Assumption | Simple meaning | Why it matters |
|---|---|---|
| Linearity | target changes roughly like a line with features | a straight-line model cannot capture strong curves |
| Homoscedasticity | error spread is roughly similar across prediction levels | if errors grow with salary, uncertainty is not constant |
| Independence | rows/errors should not strongly depend on each other | duplicate or grouped rows can make confidence misleading |
| Normality of errors | errors are roughly bell-shaped | mainly matters for inference and confidence intervals |

For prediction, a model can still be useful even if assumptions are imperfect. For statistical interpretation, assumptions matter more.

#### Logistic Regression: Sigmoid, Log-Odds, MLE, Binary and Multinomial

Logistic regression first computes a score:

```text
score = intercept + w1*x1 + w2*x2 + ...
```

The **sigmoid** converts that score into a probability:

```text
probability = 1 / (1 + exp(-score))
```

**Log-odds** means the model is linear on this scale:

```text
log(p / (1 - p)) = intercept + weights * features
```

**Maximum Likelihood Estimation (MLE)** means the model chooses weights that make the observed labels most likely.

Binary vs multinomial:

- **Binary logistic regression:** two classes, such as left/stayed.
- **One-vs-rest:** train one binary classifier per class.
- **Softmax / multinomial:** one model produces probabilities across all classes together.

#### KNN Distance Metrics

KNN depends on distance, so the distance metric changes the model.

| Metric | Formula idea | Best intuition |
|---|---|---|
| Euclidean | straight-line distance | physical distance on a map |
| Manhattan | sum of absolute differences | city-block movement |
| Minkowski | general family containing Euclidean and Manhattan | adjustable distance rule |
| Cosine | angle/direction similarity | text vectors or direction matters more than size |

Because salary can be thousands and tenure is small, KNN needs scaling. Otherwise salary dominates distance.

#### Naive Bayes, Bayes Theorem, and Laplace Smoothing

Bayes theorem:

```text
P(class | evidence) = P(evidence | class) * P(class) / P(evidence)
```

Naive Bayes assumes features are conditionally independent inside each class. This is often not perfectly true, but it makes the model simple and fast.

**Laplace smoothing** prevents zero probabilities.

Example:

```text
If a word never appeared in class A during training,
without smoothing P(word | class A) becomes 0.
Then the whole class probability may become 0.
```

Laplace smoothing adds a small count so unseen items do not destroy the probability.

#### SVM: Margin, Hard/Soft Margin, C, Slack, Kernel Trick, Mercer

SVM tries to maximize the margin between classes.

- **Support vectors:** closest points to the boundary.
- **Hard margin:** no mistakes allowed; works only when perfectly separable.
- **Soft margin:** allows some mistakes.
- **Slack variables:** measure how much a point violates the margin.
- **C:** controls mistake penalty.

```text
small C -> allows more mistakes, wider/smoother margin
large C -> punishes mistakes strongly, can overfit
```

The **kernel trick** lets SVM create curved boundaries without manually creating all transformed features.

Mercer's theorem gives the mathematical condition for a valid kernel. For learning practice, use standard kernels: linear, polynomial, RBF, sigmoid.

#### Decision Trees: Gini, Entropy, Information Gain, Variance Reduction, Pruning

Decision trees split data by asking questions.

Classification split criteria:

- **Gini impurity:** how mixed the classes are.
- **Entropy:** uncertainty in class labels.
- **Information gain:** how much a split reduces uncertainty.

Regression split criterion:

- **Variance reduction:** how much a split reduces target spread.

Stopping and pruning:

- `max_depth`: limits tree levels.
- `min_samples_split`: minimum rows needed to split.
- `min_samples_leaf`: minimum rows in a final leaf.
- **Minimal cost-complexity pruning:** removes branches that add complexity without enough improvement.

#### Bagging, OOB, Random Forest, Extra Trees

**Bagging** trains many models on bootstrap samples.

**OOB error** means out-of-bag error. Since each bootstrap sample leaves out some rows, those left-out rows can estimate performance for trees that did not train on them.

Random Forest adds feature randomness:

```text
each tree sees a row sample
each split sees only some features
```

Extra Trees adds more randomness by choosing split thresholds more randomly. This can reduce variance and speed training.

#### Boosting: AdaBoost and Gradient Boosting

AdaBoost:

- trains weak learners sequentially
- increases weight on rows predicted incorrectly
- combines learners using estimator weights
- is related to exponential loss

Gradient Boosting:

- starts with a simple prediction
- computes residual-like errors
- trains the next learner to fix those errors
- uses learning rate to control correction strength

```text
small learning_rate + more trees = slower but often better controlled
large learning_rate = faster but can overfit or become unstable
```

#### Modern Boosting: XGBoost, LightGBM, CatBoost

XGBoost:

- regularized objective with L1/L2 on leaf weights
- second-order Taylor approximation of loss
- handles sparse/missing values with learned default directions
- uses efficient block/parallel structure

LightGBM:

- histogram-based split search
- leaf-wise best-first growth
- **GOSS:** keeps high-gradient rows and samples low-gradient rows
- **EFB:** bundles sparse mutually exclusive features

CatBoost:

- native categorical feature handling
- ordered boosting to reduce target leakage
- symmetric/oblivious trees for fast inference

#### Stacking and Blending

Stacking uses out-of-fold predictions from base models to train a meta-model. This avoids training the meta-model on predictions from models that already saw the same rows.

Blending is simpler: train base models, predict on a holdout set, and train a blender/meta-model on that holdout prediction table.


### Algorithm-Specific Hyperparameters

A **parameter** is learned by the model during training.

Example:

```text
Linear regression learns the slope and intercept.
```

A **hyperparameter** is chosen before training.

Example:

```text
KNN does not learn k. We choose k before training.
```

Hyperparameters control the behavior of the algorithm. They often control the tradeoff between underfitting and overfitting.

#### General tuning idea

```text
small/simple setting -> may underfit
large/complex setting -> may overfit
balanced setting -> better generalization
```

We choose hyperparameters using validation or cross-validation, not by looking at the final test set.

Correct workflow:

```text
choose algorithm -> choose metric -> choose candidate settings -> cross-validate -> choose best setting -> final test evaluation
```

#### Linear and Logistic Regression

Important hyperparameters:

| Hyperparameter | Meaning | If too weak | If too strong |
|---|---|---|---|
| regularization strength | controls coefficient size | coefficients may become too large | model may become too simple |
| penalty | L1, L2, ElasticNet | wrong penalty may keep noise | wrong penalty may remove useful features |
| `C` in logistic regression | inverse regularization strength | large `C` can overfit | small `C` can underfit |

Remember:

```text
For LogisticRegression, smaller C means stronger regularization.
```

#### Polynomial Regression

Important hyperparameter:

| Hyperparameter | Meaning |
|---|---|
| `degree` | how much curve complexity is allowed |

- degree `1`: straight line
- degree `2`: simple curve
- very high degree: can overfit badly

#### KNN

Important hyperparameters:

| Hyperparameter | Meaning | Small value | Large value |
|---|---|---|---|
| `n_neighbors` | number of nearby rows used | sensitive to noise | smoother, may miss local pattern |
| distance metric | how closeness is measured | depends on data | depends on data |
| weights | equal vote or closer neighbors matter more | local influence | smoother influence |

KNN almost always needs scaling because salary values are much larger than years.

#### Naive Bayes

Important hyperparameter:

| Hyperparameter | Meaning |
|---|---|
| smoothing / `alpha` | prevents zero probability for unseen values |

For text classification, smoothing is important because a word may appear in test data even if it was not seen in training.

#### SVM

Important hyperparameters:

| Hyperparameter | Meaning | If too small | If too large |
|---|---|---|---|
| `C` | penalty for mistakes | wider margin, more mistakes allowed | stricter boundary, may overfit |
| `kernel` | boundary shape | linear may underfit curved data | flexible kernels may overfit |
| `gamma` for RBF | how local the influence is | boundary too smooth | boundary too wiggly |

Simple memory:

```text
C controls mistake tolerance.
gamma controls how tightly the boundary bends around points.
```

#### Decision Tree

Important hyperparameters:

| Hyperparameter | Meaning | If too small | If too large |
|---|---|---|---|
| `max_depth` | maximum levels in tree | underfits | overfits |
| `min_samples_split` | rows needed to split a node | fewer splits | many tiny splits |
| `min_samples_leaf` | rows required in final leaf | smoother leaves | tiny leaves can memorize |
| pruning `ccp_alpha` | removes weak branches | too much pruning underfits | too little pruning overfits |

A deep tree can memorize the training dataset. Depth control is one of the most important tree settings.

#### Random Forest and Extra Trees

Important hyperparameters:

| Hyperparameter | Meaning |
|---|---|
| `n_estimators` | number of trees |
| `max_features` | number of features considered at each split |
| `max_depth` | maximum depth of each tree |
| `min_samples_leaf` | minimum rows per leaf |
| `bootstrap` | whether trees sample rows with replacement |

More trees usually make the forest more stable, but training becomes slower. Tree depth and leaf size still control overfitting.

#### Boosting Models

Important hyperparameters:

| Hyperparameter | Meaning |
|---|---|
| `n_estimators` | number of boosting rounds |
| `learning_rate` | how much each new model contributes |
| tree depth / leaves | complexity of each weak learner |
| subsampling | trains on part of data each round to reduce overfitting |
| regularization | controls leaf weights or model complexity |

Key relationship:

```text
smaller learning_rate usually needs more n_estimators
larger learning_rate learns faster but can overfit or become unstable
```

#### Grid Search, Randomized Search, and Bayesian Optimization

- **Grid Search:** tries every setting combination in a fixed list. Simple but can be slow.
- **Randomized Search:** tries random combinations. Useful when there are many settings.
- **Bayesian Optimization:** uses previous results to choose smarter next trials. Useful when training is expensive.

For students, the important idea is not the search tool name. The important idea is:

```text
Try settings on validation data, choose based on validation score, and keep the final test set untouched.
```


In [13]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score

X = df[['Years at Company', 'Monthly Salary']]
y = df['Left_Binary']
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# 1. KNN: compare different k values and distance metrics.
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

knn_grid = {
    'model__n_neighbors': [1, 3, 5],
    'model__metric': ['euclidean', 'manhattan']
}

knn_search = GridSearchCV(knn_pipe, param_grid=knn_grid, scoring='accuracy', cv=cv)
knn_search.fit(X, y)

print('KNN hyperparameter tuning example')
print('Best KNN parameters:', knn_search.best_params_)
print(f'Best KNN cross-validation accuracy: {knn_search.best_score_:.3f}')

display(pd.DataFrame(knn_search.cv_results_)[[
    'param_model__n_neighbors',
    'param_model__metric',
    'mean_test_score',
    'rank_test_score'
]].sort_values('rank_test_score'))

# 2. Decision Tree: show how max_depth changes validation score.
tree_rows = []
for depth in [1, 2, 3, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(tree, X, y, cv=cv, scoring='accuracy')
    tree_rows.append({
        'max_depth': 'None' if depth is None else depth,
        'mean_cv_accuracy': scores.mean(),
        'fold_scores': [round(s, 3) for s in scores]
    })

print('\nDecision Tree max_depth comparison')
display(pd.DataFrame(tree_rows))

# 3. Logistic Regression: show how C changes regularization strength.
log_rows = []
for c_value in [0.01, 0.1, 1, 10]:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(C=c_value, max_iter=1000, random_state=42))
    ])
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    log_rows.append({
        'C': c_value,
        'regularization_meaning': 'stronger' if c_value < 1 else 'weaker' if c_value > 1 else 'baseline',
        'mean_cv_accuracy': scores.mean(),
        'fold_scores': [round(s, 3) for s in scores]
    })

print('\nLogistic Regression C comparison')
display(pd.DataFrame(log_rows))


KNN hyperparameter tuning example
Best KNN parameters: {'model__metric': 'euclidean', 'model__n_neighbors': 1}
Best KNN cross-validation accuracy: 0.806


,param_model__n_neighbors,param_model__metric,mean_test_score,rank_test_score
0,1,euclidean,0.805556,1
3,1,manhattan,0.805556,1
1,3,euclidean,0.694444,3
2,5,euclidean,0.694444,3
4,3,manhattan,0.694444,3
5,5,manhattan,0.694444,3



Decision Tree max_depth comparison


,max_depth,mean_cv_accuracy,fold_scores
0,1,0.638889,"[0.25, 1.0, 0.667]"
1,2,0.888889,"[1.0, 1.0, 0.667]"
2,3,0.722222,"[0.5, 1.0, 0.667]"
3,None,0.722222,"[0.5, 1.0, 0.667]"



Logistic Regression C comparison


,C,regularization_meaning,mean_cv_accuracy,fold_scores
0,0.01,stronger,0.694444,"[0.75, 0.667, 0.667]"
1,0.10,stronger,0.694444,"[0.75, 0.667, 0.667]"
2,1.00,baseline,0.805556,"[0.75, 1.0, 0.667]"
3,10.00,weaker,0.805556,"[0.75, 1.0, 0.667]"
